<a href="https://colab.research.google.com/github/olihile84-tech/exercise-syntax-variables-and-numbers/blob/main/My_own_pandas_trade_calculator_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

trades = {
    'pair': ['EURUSD', 'GBPJPY', 'EURUSD', 'XAUUSD', 'GBPJPY'],
    'session': ['London', 'Tokyo', 'New York', 'London', 'New York'],
    'entry': [1.1000, 185.00, 1.0950, 1920.00, 184.50],
    'exit': [1.1050, 184.50, 1.0900, 1935.00, 185.20],
    'sl': [1.0980, 185.20, 1.0970, 1915.00, 184.20],
    'result': ['win', 'loss', 'loss', 'win', 'win']
}

journal = pd.DataFrame(trades)
journal

,pair,session,entry,exit,sl,result
0,EURUSD,London,1.100,1.105,1.098,win
1,GBPJPY,Tokyo,185.000,184.500,185.200,loss
2,EURUSD,New York,1.095,1.090,1.097,loss
3,XAUUSD,London,1920.000,1935.000,1915.000,win
4,GBPJPY,New York,184.500,185.200,184.200,win


In [6]:
journal ['pips'] = (journal['exit']-journal['exit'])*10000
print(journal[['pair', 'pips']])

     pair  pips
0  EURUSD   0.0
1  GBPJPY   0.0
2  EURUSD   0.0
3  XAUUSD   0.0
4  GBPJPY   0.0


In [7]:
def calculate_pip(row):
  diff = row['exit']-row['entry']
  if 'JPY' in row ['pair']:
    return diff * 100
  elif 'XAU' in row ['pair']:
    return  diff * 10
  else:
    return diff * 10000

journal['pips']=journal.apply(calculate_pip,axis=1)
print(journal[['pair','session','result','pips']])

     pair   session result   pips
0  EURUSD    London    win   50.0
1  GBPJPY     Tokyo   loss  -50.0
2  EURUSD  New York   loss  -50.0
3  XAUUSD    London    win  150.0
4  GBPJPY  New York    win   70.0


In [8]:
def calculate_rr(row):
  risk = abs(row['entry']-row['sl'])
  reward = abs(row['exit']-row['entry'])
  if risk == 0:
    return 0
  else:
    return reward/risk

journal['rr']=journal.apply(calculate_rr,axis=1)
print(journal[['pair','session','result','rr']])



     pair   session result        rr
0  EURUSD    London    win  2.500000
1  GBPJPY     Tokyo   loss  2.500000
2  EURUSD  New York   loss  2.500000
3  XAUUSD    London    win  3.000000
4  GBPJPY  New York    win  2.333333


In [9]:
total_trades = len (journal)
win = len(journal[journal['result']== 'win'])
win_rate = (win/total_trades)*100

avg_rr = journal['rr'].mean()

print(f'Total Trades: {total_trades}')
print(f'Win: {win}')
print(f'Win Rate: {win_rate:.2f}%')
print(f'Average RR: {round(avg_rr,2)}')

Total Trades: 5
Win: 3
Win Rate: 60.00%
Average RR: 2.57


In [10]:
best_pair = journal.groupby ('pair')['rr'].mean().sort_values(ascending=False)
print("Average RR per pair:")
print(best_pair)

Average RR per pair:
pair
XAUUSD    3.000000
EURUSD    2.500000
GBPJPY    2.416667
Name: rr, dtype: float64


In [11]:
journal.to_csv('trading_journal.cvs', index=False)
print("Journal saved!")

Journal saved!


In [12]:
session_stats = journal.groupby('session')['result'].apply(
    lambda x: (x=='win').sum()/len(x)*100
).round(2)
print("Wins rate per session:")
print(session_stats)

Wins rate per session:
session
London      100.0
New York     50.0
Tokyo         0.0
Name: result, dtype: float64


In [13]:
ACCOUNT_BALANCE = 10000
RISK_PERCENT = 0.25
PIP_VALUE = 10

def calculate_position_size(row):
  cash_at_risk = ACCOUNT_BALANCE*(RISK_PERCENT/100)
  sl_pips = abs(row['entry']- row['sl'])

  if 'JPY' in row['pair']:
    sl_pips = sl_pips*100
  elif 'XAU' in row['pair']:
    sl_pips = sl_pips*10
  else:
    sl_pips = sl_pips*10000

  lot_size= round(cash_at_risk/(sl_pips*PIP_VALUE),2)
  return lot_size

journal['lot_size']= journal.apply(calculate_position_size, axis=1)
journal['cash_at_risk']= ACCOUNT_BALANCE*(RISK_PERCENT/100)

print(journal[['pair','session','result','rr','lot_size','cash_at_risk']])


     pair   session result        rr  lot_size  cash_at_risk
0  EURUSD    London    win  2.500000      0.12          25.0
1  GBPJPY     Tokyo   loss  2.500000      0.13          25.0
2  EURUSD  New York   loss  2.500000      0.12          25.0
3  XAUUSD    London    win  3.000000      0.05          25.0
4  GBPJPY  New York    win  2.333333      0.08          25.0


In [14]:
ACCOUNT_BALANCE = 10000
RISK_PERCENT = 0.25
PIP_VALUE = 10

def calculate_pnl(row):
  cash_at_risk = ACCOUNT_BALANCE * (RISK_PERCENT/100)
  if row['result']== 'win' :
    return round (cash_at_risk*row['rr'],2)
  else:
    return round(-cash_at_risk,2)

journal['pnl']= journal.apply(calculate_pnl, axis=1)

total_pnl = journal['pnl'].sum()
print(journal[['pair','session','rr','pnl']])
print(f'\nTotal P&L: ${total_pnl}')
print(f'Starting Balance: ${ACCOUNT_BALANCE}')
print(f'Final Balance: ${ACCOUNT_BALANCE + total_pnl}')

     pair   session        rr    pnl
0  EURUSD    London  2.500000  62.50
1  GBPJPY     Tokyo  2.500000 -25.00
2  EURUSD  New York  2.500000 -25.00
3  XAUUSD    London  3.000000  75.00
4  GBPJPY  New York  2.333333  58.33

Total P&L: $145.82999999999998
Starting Balance: $10000
Final Balance: $10145.83
